# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and fields' IDs.

We will enumerate the record sets in the dataset, each field in every record set, and their respective `@id`s for reference in future code.

In [ ]:
# Utility functions to print record sets and fields with their @id

def print_record_sets(dataset):
    print("Available Record Sets:")
    for rs in dataset.metadata.record_sets:
        print(f"  - name: {rs.name}, @id: {rs.id}")
        print("    Fields:")
        for field in rs.fields:
            print(f"      - name: {field.name}, @id: {field.id}, dataType: {field.data_type}")

print_record_sets(dataset)

## 3. Data Extraction
Let's extract the data from each record set into pandas DataFrames for analysis.

We will demonstrate extraction using the available record sets and refer to all entities by their `@id` for reproducibility.

In [ ]:
# Collect all record set @id's
record_sets = [rs.id for rs in dataset.metadata.record_sets]
print("Record Set @id's:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    # For each record set, extract all records as dicts and load into a DataFrame
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create DataFrame if there are records
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")

# For demo, print the columns of the first available record set
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns for record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No record set dataframes loaded. Check schema or data accessibility.")

## 4. Exploratory Data Analysis (EDA)
Let's apply some EDA steps such as filtering, normalization, and grouping on numeric fields.

**Note:** Remember to use the correct `@id` for the record set and the fields referenced below. Modify the fields if necessary based on the printout above.

In [ ]:
import numpy as np

# Choose record set for EDA - using the first available for demo purposes
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Identify numeric fields. We'll pick the first numeric one for demo.
    rs_obj = next(rs for rs in dataset.metadata.record_sets if rs.id == record_set_id)
    numeric_fields = [f for f in rs_obj.fields if f.data_type in ('Integer', 'Float', 'Number')]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        # The columns use field names; map id to the column name
        numeric_field_name = numeric_fields[0].name
        print(f"Using numeric field: {numeric_field_name} (id: {numeric_field_id})")

        # Convert column to numeric if not already
        df[numeric_field_name] = pd.to_numeric(df[numeric_field_name], errors='coerce')
        # Filter by an example threshold (median for demo)
        threshold = df[numeric_field_name].median()
        filtered_df = df[df[numeric_field_name] > threshold]
        print(f"Filtered records with {numeric_field_name} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_name}_normalized"] = (
            filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()
        ) / filtered_df[numeric_field_name].std()
        print(f"Normalized {numeric_field_name} for filtered records:")
        display(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

        # Group by a categorical/text field if available
        group_fields = [f for f in rs_obj.fields if f.data_type in ('Text', 'String')]
        if group_fields:
            group_field_id = group_fields[0].id
            group_field_name = group_fields[0].name
            if group_field_name in filtered_df.columns:
                grouped_df = (
                    filtered_df.groupby(group_field_name)[numeric_field_name].mean().to_frame()
                )
                print(f"Grouped mean of {numeric_field_name} by {group_field_name}:")
                display(grouped_df.head())
    else:
        print(f"No numeric fields found in record set {record_set_id} for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram of numeric field
if dataframes and numeric_fields:
    plt.figure(figsize=(8, 6))
    sns.histplot(df[numeric_field_name].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_name}")
    plt.xlabel(numeric_field_name)
    plt.ylabel("Count")
    plt.show()

    # Optionally, boxplot by group field
    if group_fields and group_field_name in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_name, y=numeric_field_name, data=df)
        plt.title(f"Boxplot of {numeric_field_name} by {group_field_name}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualizations.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset by loading the schema and records using the `mlcroissant` library. We reviewed the structure, extracted data using record set and field `@id`s, and performed basic analysis and plotting.

**Key Observations:**
- The dataset contains detailed mass spectrometry results for extracellular vesicle proteins from human mast cells.
- We identified numeric and categorical fields, filtered and normalized values, and visualized distributions and grouped summaries.

Continue with further domain-specific analyses as required. For more information, refer to the [Croissant documentation](https://mlcommons.github.io/croissant/) and dataset schema.